# 🔴 Solution: Polygon Triangulation (Ear-Clipping)

**Primitive:** `(K, 1, 2)` vs `(1, K, 2)` broadcast for the point-in-triangle inner test

**Reduction:** at each step, vertex B is an ear iff `cross(B−A, C−A) > 0` AND the `(K, K)` matrix of `>= 0` point-in-triangle tests (all K active vertices against all K candidate ear triangles) has no True in column j outside rows j−1, j, j+1 (excluded via mask — the ear's own vertices trivially satisfy `>= 0`).

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# primitive: (K,1,2) vs (1,K,2) broadcast for batch point-in-triangle

def triangulate_polygon(vertices):
    n = len(vertices)
    idx = np.arange(n)
    triangles = []

    while len(idx) > 3:
        K = len(idx)
        v  = vertices[idx]              # (K, 2) active vertices
        ki = np.arange(K)

        A = v[(ki - 1) % K]             # (K, 2) previous neighbour
        B = v[ki]                        # (K, 2) candidate ear vertex
        C = v[(ki + 1) % K]             # (K, 2) next neighbour

        # ── Step 1: convexity — cross(B-A, C-A) > 0 ──────────────────────
        ba = B - A; ca = C - A
        convex = ba[:, 0] * ca[:, 1] - ba[:, 1] * ca[:, 0] > 0   # (K,)

        # ── Step 2: vectorised point-in-triangle ──────────────────────────
        # P[i, 0, :] = active vertex i as test point
        # Aj[0, j, :] = A of ear candidate j  (and similarly Bj, Cj)
        P  = v[:, None, :]              # (K, 1, 2)
        Aj = A[None, :, :]              # (1, K, 2)
        Bj = B[None, :, :]              # (1, K, 2)
        Cj = C[None, :, :]              # (1, K, 2)

        def ecross(o, a, p):
            d = a - o; q = p - o
            return d[..., 0] * q[..., 1] - d[..., 1] * q[..., 0]

        # >= 0 catches vertices that land exactly on an ear edge (non-strict
        # inside), which the strict > 0 test would miss in non-general-position
        # inputs.  The ear's own A/B/C all satisfy >= 0, so we exclude them.
        inside = (
            (ecross(Aj, Bj, P) >= 0) &
            (ecross(Bj, Cj, P) >= 0) &
            (ecross(Cj, Aj, P) >= 0)
        )                               # (K, K)

        # Exclude each ear's own three vertices from the inside check
        exclude = np.zeros((K, K), dtype=bool)
        exclude[(ki - 1) % K, ki] = True   # A of ear j
        exclude[ki, ki]           = True   # B
        exclude[(ki + 1) % K, ki] = True   # C

        has_vertex_inside = (inside & ~exclude).any(axis=0)   # (K,)
        is_ear = convex & ~has_vertex_inside

        # Clip the first ear found
        j = int(np.argmax(is_ear))
        triangles.append([int(idx[(j - 1) % K]), int(idx[j]), int(idx[(j + 1) % K])])
        idx = np.delete(idx, j)

    triangles.append(idx.tolist())
    return np.array(triangles, dtype=int)

In [ ]:
# 🔍 Verify solution
L = np.array([[0.,0.],[4.,0.],[4.,2.],[2.,2.],[2.,4.],[0.,4.]])
tris = triangulate_polygon(L)
print("triangles:", tris)
print("shape:", tris.shape)  # expect (4, 3)

def tri_area(v, t):
    A, B, C = v[t[0]], v[t[1]], v[t[2]]
    return 0.5 * abs((B-A)[0]*(C-A)[1] - (B-A)[1]*(C-A)[0])

total = sum(tri_area(L, t) for t in tris)
print(f"total area: {total:.4f}  (expected 12.0)")

In [ ]:
# ✅ Inline test suite
import numpy as np, time

def tri_area(v, t):
    A, B, C = v[t[0]], v[t[1]], v[t[2]]
    return 0.5 * abs((B-A)[0]*(C-A)[1] - (B-A)[1]*(C-A)[0])

def shoelace(v):
    x, y = v[:,0], v[:,1]
    return 0.5 * abs(np.sum(x * np.roll(y,-1) - np.roll(x,-1) * y))

# ── Test 1: trivial triangle ───────────────────────────────────────────────
tri = np.array([[0.,0.],[1.,0.],[0.,1.]])
r1 = triangulate_polygon(tri)
assert r1.shape == (1, 3), f"Shape: {r1.shape}"
assert set(r1[0]) == {0, 1, 2}
print("Test 1 passed: trivial triangle")

# ── Test 2: convex square ─────────────────────────────────────────────────
sq = np.array([[0.,0.],[2.,0.],[2.,2.],[0.,2.]])
r2 = triangulate_polygon(sq)
assert r2.shape == (2, 3)
assert abs(sum(tri_area(sq, t) for t in r2) - 4.0) < 1e-9
print("Test 2 passed: convex square")

# ── Test 3: concave L-shape ───────────────────────────────────────────────
L = np.array([[0.,0.],[4.,0.],[4.,2.],[2.,2.],[2.,4.],[0.,4.]])
r3 = triangulate_polygon(L)
assert r3.shape == (4, 3), f"Shape: {r3.shape}"
assert abs(sum(tri_area(L, t) for t in r3) - 12.0) < 1e-9
print("Test 3 passed: concave L-shape")

# ── Test 4: random 12-gon — area preserved ────────────────────────────────
rng = np.random.default_rng(17)
angles = np.sort(rng.uniform(0, 2*np.pi, 12))
v12 = np.stack([np.cos(angles), np.sin(angles)], axis=1)
r4 = triangulate_polygon(v12)
assert r4.shape == (10, 3)
assert abs(sum(tri_area(v12, t) for t in r4) - shoelace(v12)) < 1e-9
print("Test 4 passed: 12-gon area preserved")

# ── Test 5: n=200, must finish < 5s ───────────────────────────────────────
rng = np.random.default_rng(31)
angles200 = np.sort(rng.uniform(0, 2*np.pi, 200))
v200 = np.stack([np.cos(angles200), np.sin(angles200)], axis=1)
t0 = time.time()
r5 = triangulate_polygon(v200)
elapsed = time.time() - t0
assert r5.shape == (198, 3)
assert abs(sum(tri_area(v200, t) for t in r5) - shoelace(v200)) < 1e-6
assert elapsed < 5.0, f"Too slow: {elapsed:.2f}s"
print(f"Test 5 passed: n=200 in {elapsed:.3f}s")

print("\nAll tests passed!")